# Lab 28 · Let the gateway choose the model for each request

Put `gpt-5-mini` and `gpt-5.5` behind one alias, `auto`, and let Microsoft Foundry Model Router pick one of them for
each prompt. Send the same ten prompts with the **Balanced** and **Cost** objectives, then let a caller ask for
**Quality** in its own requests, and chart which model served each prompt and how many tokens each model used. This
notebook is the Python track of the [lab guide](README.md): it deploys the same
[infra/auto-alias.bicep](infra/auto-alias.bicep) as the Bicep + azd track, and sends the prompts of
[infra/prompts.json](infra/prompts.json), like [infra/route-prompts.sh](infra/route-prompts.sh).

**Check the models → Deploy the auto alias → Run the prompts with Balanced → Switch to Cost → Allow a Quality override → Chart the split → Narrow the pool → Clean up**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Backend model `foundry-models/gpt-5-mini` | The lower-cost model from notebook 02. |
| Foundry deployment `gpt-5.5` | The higher-quality model. The notebook deploys it next to `gpt-5-mini` if it's missing, with [infra/large-deployment.bicep](infra/large-deployment.bicep). |
| Backend model `foundry-models/gpt-5-5` | Registers the `gpt-5.5` deployment in the gateway, under the name that the portal's import gives it. |
| Alias `auto` | Lists both backend models with their `selectorModelName`, runs priority before Model Router, and sets the objective and the per-request overrides. |
| Runtime key `app-first-call` | The key from notebook 02. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, Bicep deployments, keys, and retries. |

![Three clients call the auto alias with a simple prompt, a hard prompt, and an objective override. The gateway applies priority, then the objective, then Model Router picks gpt-5-mini or gpt-5.5](../assets/diagrams/auto-model-selection.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource group) and create deployments in
  the Foundry account.
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  deploys Bicep with `az deployment group create`, and the CLI installs Bicep the first time it needs it.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) and
  [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb) have run, so the gateway publishes
  the `gpt-5-mini` model and can call your Foundry account.
- Quota for `gpt-5.5` in the Foundry account's region: the notebook deploys 20,000 tokens per minute of Global
  Standard capacity if the account has no `gpt-5.5` deployment yet.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it updates the same resources.
The three prompt runs send 30 requests and use about 15,000 output tokens.

In [ ]:
%pip install -q --upgrade azure-identity requests openai matplotlib

### 0. Initialize variables

The defaults match notebooks 01 and 02, so `utils.Lab` finds the gateway without edits. `SMALL` is the backend model
that notebook 02 published, and `LARGE` is the deployment that this lab adds. Each answer stops at 1,000 output
tokens to keep the runs short and cheap, which doesn't change the model that Model Router picks.

In [ ]:
import json
import sys
import time
from collections import Counter
from IPython.display import display

sys.path.insert(0, "../shared")   # helpers and brand styles that every lab notebook shares
import brand
import utils

lab = utils.Lab("28-auto-model-selection")
PROVIDER   = "foundry-models"   # model provider from notebook 02
SMALL      = "gpt-5-mini"       # lower-cost backend model from notebook 02, and its Model Router name
LARGE      = "gpt-5.5"          # higher-quality deployment and its Model Router name
ALIAS      = "auto"             # the name that clients send in "model"
MAX_TOKENS = 1000               # output cap for each answer
PROMPTS    = json.load(open("infra/prompts.json"))   # five simple prompts and five hard ones

### 1. Check the models

The cell confirms that notebook 02 published `gpt-5-mini`, then looks for a `gpt-5.5` deployment in the Foundry
account. If there's none, it deploys one with [infra/large-deployment.bicep](infra/large-deployment.bicep), and the
clean-up step deletes it again.

In [ ]:
lab.require(f"{lab.workspace_id}/modelProviders/{PROVIDER}/models/{SMALL}", "Run 02-first-model-call.ipynb first.")
display(brand.note(f"Model {PROVIDER}/{SMALL} exists", "ok"))

deployment_id = f"{lab.foundry_id}/deployments/{LARGE}"
created_deployment = lab.arm("GET", deployment_id, api_version=utils.COGNITIVE_API) is None
if created_deployment:
    lab.deploy("infra/large-deployment.bicep", resource_group=lab.foundry_resource_group,
               foundryName=lab.foundry_account, modelName=LARGE)
deployment = lab.arm("GET", deployment_id, api_version=utils.COGNITIVE_API)
display(brand.info(f"Foundry deployment {LARGE}", [
    ("Model", f"{deployment['properties']['model']['name']} {deployment['properties']['model']['version']}"),
    ("SKU", f"{deployment['sku']['name']}, capacity {deployment['sku']['capacity']}"),
    ("Created by this notebook", "yes" if created_deployment else "no, it already existed")], sub=lab.foundry_account))

### 2. Deploy the auto alias

[infra/auto-alias.bicep](infra/auto-alias.bicep) registers the `gpt-5.5` deployment as the backend model `gpt-5-5`
and writes the alias `auto`:

- `backendModels` lists both backend models at priority `0` and weight `1`, each with a `selectorModelName`: the
  unversioned model name that Model Router knows it by.
- `selectionOrder` is `["priority", "autoModelSelection"]`: priority keeps the best available priority group, then
  Model Router picks one model in it.
- `autoModelSelection` holds the `objective`, **Balanced** for now, and `allowedRequestOverrides`, empty: callers
  can't change the selection yet.

A new alias takes up to a minute to reach every gateway instance, so the cell sends a short prompt until three
answers in a row report the **Balanced** objective.

In [ ]:
from openai import OpenAI, APIStatusError

AUTO_HEADERS = ("x-ms-served-model", "x-ms-model-router-model", "x-ms-model-router-mode")

def deploy_alias(objective, overrides=()):
    """Deploy the auto alias with an objective and the fields that callers may override. Show its settings."""
    lab.deploy("infra/auto-alias.bicep", gatewayName=lab.gateway_name, providerName=PROVIDER,
               smallModelName=SMALL, smallSelectorModelName=SMALL, foundryAccountId=lab.foundry_id,
               largeDeploymentName=LARGE, largeModelName=LARGE, aliasName=ALIAS,
               objective=objective, allowedRequestOverrides=list(overrides))
    p = lab.arm("GET", f"{lab.workspace_id}/aliases/{ALIAS}")["properties"]
    rows = [(brand.mono(m["resourceId"].rsplit("/", 1)[-1]),
             f"selector {m.get('selectorModelName')} · priority {m['priority']} · weight {m['weight']}")
            for m in p["backendModels"]]
    rows += [("Routing order", brand.mono(", ".join(p.get("selectionOrder", ["priority"])))),
             ("Objective", p["autoModelSelection"]["objective"]),
             ("Caller overrides", ", ".join(p["autoModelSelection"].get("allowedRequestOverrides") or []) or "none")]
    display(brand.info(f"Alias {ALIAS}", rows, sub=lab.gateway_name))
    return p

def ask(prompt, objective=None, models=None, client=None):
    """Send one prompt to the alias, with an optional auto_model_selection override. Return the raw response."""
    override = {k: v for k, v in (("objective", objective), ("models", models)) if v}
    return (client or sdk).chat.completions.with_raw_response.create(
        model=ALIAS, max_completion_tokens=MAX_TOKENS, messages=[{"role": "user", "content": prompt}],
        extra_body={"auto_model_selection": override} if override else None)

def wait_for_mode(mode, objective=None, minutes=5):
    """Send a short prompt until three answers in a row report the expected x-ms-model-router-mode."""
    status, start, streak = brand.live(), time.time(), 0
    while streak < 3:
        try:
            seen = ask("What is 12 times 12?", objective=objective, client=strict).headers.get("x-ms-model-router-mode")
        except APIStatusError as error:
            seen = f"HTTP {error.status_code}"
        streak = streak + 1 if seen == mode else 0
        if streak == 0:
            elapsed = int(time.time() - start)
            if elapsed > minutes * 60:
                raise TimeoutError(f"The alias still answers {seen} instead of the {mode} objective.")
            status.update(brand.note(f"Got {seen} · waiting for the {mode} objective to apply · {elapsed} s", "wait"))
            time.sleep(15)
    status.update(brand.note(f"The alias answers with x-ms-model-router-mode: {mode}", "ok"))

api_key, _ = lab.api_key("app-first-call")
display(brand.key("app-first-call", api_key))
base_url = f"{lab.gateway_url}/default/models/openai/v1"
sdk = OpenAI(base_url=base_url, api_key=api_key, max_retries=5)      # retries 429 and 5xx answers
strict = OpenAI(base_url=base_url, api_key=api_key, max_retries=0)   # shows every error as it comes

deploy_alias("Balanced")
wait_for_mode("balanced")

### 3. Send a simple and a hard prompt

Send a yes-or-no question and a proof, and look at three response headers:

- `x-ms-served-model` names the model and version that answered.
- `x-ms-model-router-model` names the model that Model Router picked, as `name@version`.
- `x-ms-model-router-mode` names the objective that Model Router used.

In [ ]:
for item in (PROMPTS[0], PROMPTS[5]):
    raw = ask(item["prompt"])
    display(brand.http(raw.status_code, raw.headers, raw.text, f"A {item['kind']} prompt", sub=item["prompt"][:80],
                       keys=AUTO_HEADERS))

### 4. Run the prompts with Balanced

[infra/prompts.json](infra/prompts.json) holds five simple prompts and five hard ones: proofs, a scheduling puzzle,
and a system design question. The cell sends them one at a time and shows the model that served each one, with its
token usage. With **Balanced**, Model Router sends the hard prompts to `gpt-5.5` and most simple ones to
`gpt-5-mini`. Its choices for borderline prompts can change from one run to the next.

In [ ]:
results = {}

def run_prompts(label, objective=None):
    """Send the ten prompts, optionally with an objective override, and return a row for each one."""
    board = brand.Board(f"Which model serves each prompt · {label}", ["#", "Kind", "Served model", "Prompt tokens",
                        "Output tokens", "Prompt"], sub=f"{ALIAS} · " + (f"caller objective {objective}"
                        if objective else "the alias's objective"), stop=403)
    rows = []
    for i, item in enumerate(PROMPTS, 1):
        try:
            raw = ask(item["prompt"], objective=objective)
            usage = raw.parse().usage
            row = dict(kind=item["kind"], status=raw.status_code, served=raw.headers.get("x-ms-served-model"),
                       prompt_tokens=usage.prompt_tokens, output_tokens=usage.completion_tokens)
        except APIStatusError as error:
            row = dict(kind=item["kind"], status=error.status_code, served=f"HTTP {error.status_code}",
                       prompt_tokens=0, output_tokens=0)
        board.add([i, item["kind"], brand.mono(row["served"]), row["prompt_tokens"], row["output_tokens"],
                   item["prompt"][:60]], row["status"])
        rows.append(row)
    results[label] = rows
    return rows

balanced = run_prompts("Balanced")
assert all(r["status"] == 200 for r in balanced)
assert all(r["served"].startswith(LARGE) for r in balanced if r["kind"] == "hard"), "Balanced sends hard prompts to gpt-5.5."

### 5. Switch the objective to Cost

Deploy the alias again with the **Cost** objective. The apps don't change: they still send `"model": "auto"`. The
cell waits until the alias answers with `x-ms-model-router-mode: cost`, then sends the same prompts. Every prompt now
goes to `gpt-5-mini`, the hard ones too: the bill drops, and the hard answers come from the smaller model.

In [ ]:
deploy_alias("Cost")
wait_for_mode("cost")
cost = run_prompts("Cost")
assert all(r["status"] == 200 and r["served"].startswith(SMALL) for r in cost), "Cost sends every prompt to gpt-5-mini."

### 6. Let a caller ask for Quality

A caller can send an `auto_model_selection` field in the request body: its `objective` overrides the alias's
objective, and its `models` narrows the pool to some of the alias's selector models. Both are off by default, so
the first request gets `403` with `request_override_not_allowed`. The cell then deploys the alias with
`allowedRequestOverrides` set to `model` and `objective`, keeps the **Cost** objective, waits until the override
applies, and sends the prompts with the **Quality** objective in each request.

In [ ]:
try:
    ask(PROMPTS[5]["prompt"], objective="quality", client=strict)
    raise AssertionError("Expected 403: the alias doesn't allow overrides yet.")
except APIStatusError as error:
    display(brand.http(error.status_code, error.response.headers, error.response.text,
                       "An override that the alias doesn't allow", sub=ALIAS, keys=("content-type",)))
    assert error.status_code == 403 and "request_override_not_allowed" in error.response.text

deploy_alias("Cost", ["model", "objective"])
wait_for_mode("quality", objective="quality")
quality = run_prompts("Quality (caller)", objective="quality")
assert all(r["status"] == 200 and r["served"].startswith(LARGE) for r in quality), "Quality sends every prompt to gpt-5.5."

### 7. Chart the split

Each pair of bars shows one run: how many prompts each model served, and how many output tokens each model
produced. The hard prompts use most of the output tokens, so they're where the larger model's price matters.
`gpt-5-mini` spends many output tokens on hard prompts too, because its reasoning counts as output: it's the
price per token that makes **Cost** cheaper.

In [ ]:
import matplotlib.pyplot as plt

brand.use_matplotlib_style()
labels = list(results)
models = sorted({r["served"] for rows in results.values() for r in rows})
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
for axis, metric, title in ((left, None, "Prompts served"), (right, "output_tokens", "Output tokens")):
    bottom = [0] * len(labels)
    for model in models:
        values = [sum(1 if metric is None else r[metric] for r in results[label] if r["served"] == model)
                  for label in labels]
        axis.bar(labels, values, bottom=bottom, label=model)
        bottom = [b + v for b, v in zip(bottom, values)]
    axis.set_title(title)
left.legend(loc="upper left", fontsize=8)
fig.suptitle(f"Which model served the ten prompts of {ALIAS}, by objective")
plt.tight_layout()
plt.show()

rows = []
for label, items in results.items():
    for model, count in sorted(Counter(r["served"] for r in items).items()):
        mine = [r for r in items if r["served"] == model]
        rows.append([label, brand.mono(model), count, sum(r["prompt_tokens"] for r in mine),
                     sum(r["output_tokens"] for r in mine)])
display(brand.table("Requests and tokens per model", ["Objective", "Served model", "Requests", "Prompt tokens",
                    "Output tokens"], rows, sub=ALIAS))

### 8. Narrow the pool

`models` takes the alias's selector model names. Ask for **Quality**, but only from `gpt-5-mini`: the pool holds one
model, so `gpt-5-mini` answers the proof, and the response has no `x-ms-model-router-mode` header. A name that isn't
a selector model of the alias, such as the backend model name `gpt-5-5`, gets `400`.

In [ ]:
raw = ask(PROMPTS[5]["prompt"], objective="quality", models=[SMALL])
display(brand.http(raw.status_code, raw.headers, raw.text, "Quality, but only gpt-5-mini", sub=ALIAS, keys=AUTO_HEADERS))

try:
    ask("What is 12 times 12?", models=["gpt-5-5"], client=strict)
    bad = (200, "")
except APIStatusError as error:
    bad = (error.status_code, error.response.json()["error"]["message"])

display(brand.checks("Per-request overrides", [
    ("Quality from gpt-5-mini only", raw.status_code, 200, f"served by {raw.headers.get('x-ms-served-model')}"),
    ("A backend model name in models", bad[0], 400, bad[1])], sub=ALIAS))
assert raw.headers["x-ms-served-model"].startswith(SMALL) and bad[0] == 400

### 9. Clean up resources

The cell deletes the `auto` alias first, so that nothing points to `gpt-5-5`, then the `gpt-5-5` backend model. It
deletes the `gpt-5.5` deployment only if step 1 created it. The gateway, the `app-first-call` key, and `gpt-5-mini`
stay for the other labs.

In [ ]:
lab.delete(f"{lab.workspace_id}/aliases/{ALIAS}")
lab.delete(f"{lab.workspace_id}/modelProviders/{PROVIDER}/models/gpt-5-5")
if created_deployment:
    lab.delete(deployment_id, api_version=utils.COGNITIVE_API)

## Troubleshooting

- **`404` with `model_not_found` right after a deployment**: a new or changed alias takes up to a minute to reach
  every gateway instance. The cells wait for it; rerun the cell if it timed out.
- **`403` with `request_override_not_allowed`**: the alias doesn't allow the `auto_model_selection` fields that the
  request sent. Deploy it with `allowedRequestOverrides` set to `["model", "objective"]`, and wait a minute.
- **`400` that says `auto_model_selection.models must contain unique names from this Alias's configured selector
  Model pool.`**: `models` takes selector model names, such as `gpt-5-mini` and `gpt-5.5`, not backend model names.
- **`429` during a run**: a deployment reached its requests or tokens per minute. The OpenAI SDK waits and retries
  up to five times; raise the deployment's capacity if it keeps happening.
- **The deployment in step 1 fails with `InsufficientQuota`**: the Foundry account's region has no `gpt-5.5` quota
  left. Free some, or deploy with a smaller `capacity`.
- **Balanced sends a simple prompt to `gpt-5.5`**: that's Model Router's choice for a borderline prompt. Short
  answers cost little on either model.

For the other errors in this lab, see [Troubleshooting](../TROUBLESHOOTING.md#lab-28-auto-model-selection).

## Next steps

- Route by priority and weight in [Route an alias across models by priority and weight](../22-model-routing/README.md).
- Add conditions and session affinity in [Route alias requests with conditions and session affinity](../27-conditional-routing/README.md).
- Fail over to a backup in [Fail over to a backup model when a backend fails](../29-model-failover/README.md).
- Cap the spend per caller in [Enforce cost budgets with per-caller overrides](../07-cost-budgets/README.md) and its
  notebook [07-cost-budgets.ipynb](../07-cost-budgets/07-cost-budgets.ipynb).
- Docs: [Auto model selection](https://ai.gateway.azure.com/docs/models/auto-model-selection) and
  [Configure auto model selection](https://ai.gateway.azure.com/docs/models/configure-auto-model-selection).
- Read this lab on the labs site: [https://azure.github.io/aigateway/labs/28-auto-model-selection/](https://azure.github.io/aigateway/labs/28-auto-model-selection/).